### Simplified Self-attention

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [3]:
import torch
import torch.nn.functional as F

In [14]:
torch.manual_seed(42)

In [26]:
# 6 words, each a 3-dim embedding
X = torch.rand(6, 3)
X

tensor([[0.4467, 0.5117, 0.5770],
        [0.2721, 0.1786, 0.6670],
        [0.0228, 0.8261, 0.9829],
        [0.3100, 0.2266, 0.7803],
        [0.8634, 0.3633, 0.9124],
        [0.3378, 0.9443, 0.6525]])

In [27]:
# Step 1: attention SCORES — every word dotted with every word.
scores = X @ X.T
scores

tensor([[0.7943, 0.5978, 1.0000, 0.7047, 1.0980, 1.0106],
        [0.5978, 0.5509, 0.8094, 0.6453, 0.9085, 0.6958],
        [1.0000, 0.8094, 1.6491, 0.9613, 1.2166, 1.4291],
        [0.7047, 0.6453, 0.9613, 0.7563, 1.0620, 0.8278],
        [1.0980, 0.9085, 1.2166, 1.0620, 1.7100, 1.2301],
        [1.0106, 0.6958, 1.4291, 0.8278, 1.2301, 1.4316]])

In [28]:
# Step 2: attention WEIGHTS — softmax each row so it sums to 1.
weights = F.softmax(scores, dim=1)
weights

tensor([[0.1524, 0.1252, 0.1872, 0.1394, 0.2065, 0.1892],
        [0.1491, 0.1423, 0.1843, 0.1564, 0.2035, 0.1645],
        [0.1337, 0.1105, 0.2559, 0.1286, 0.1660, 0.2053],
        [0.1460, 0.1376, 0.1887, 0.1538, 0.2087, 0.1652],
        [0.1449, 0.1198, 0.1631, 0.1397, 0.2671, 0.1653],
        [0.1459, 0.1065, 0.2218, 0.1216, 0.1818, 0.2224]])

In [29]:
for i, row in enumerate(weights):
    print(f"sum of row {i+1} : {sum(row)}")

sum of row 1 : 1.0
sum of row 2 : 1.0000001192092896
sum of row 3 : 1.0
sum of row 4 : 1.0
sum of row 5 : 1.0
sum of row 6 : 1.0000001192092896


In [30]:
# Step 3: CONTEXT vectors — each output is a weighted sum of all input words.
context = weights @ X
context

tensor([[0.3919, 0.5403, 0.7762],
        [0.3892, 0.5186, 0.7771],
        [0.3482, 0.5829, 0.7881],
        [0.3906, 0.5218, 0.7797],
        [0.4308, 0.5151, 0.7845],
        [0.3690, 0.5805, 0.7791]])

### Self Attention with Trainable Parameters

In [31]:
# Input - 6 words, each a 3-dim embedding
X = torch.rand(6, 3)
X

tensor([[0.3058, 0.0114, 0.5138],
        [0.7311, 0.0659, 0.1376],
        [0.1224, 0.6094, 0.3663],
        [0.8096, 0.1008, 0.8041],
        [0.9603, 0.2967, 0.5654],
        [0.2735, 0.9771, 0.5435]])

In [32]:
# Step 1 - Define the query, key and value metrics
w_q = torch.rand(3, 3)
w_k = torch.rand(3, 3)
w_v = torch.rand(3, 3)

print(f"dimension of w_q: {w_q.size()}")

dimension of w_q: torch.Size([3, 3])


In [33]:
# Step 2 - Produce the Q, K and V
Q = X @ w_q
K = X @ w_k
V = X @ w_v

print(f"dimension of Q is : {Q.size()}")

dimension of Q is : torch.Size([6, 3])


In [34]:
# Step 3 - Calculate the attention scores
attention_scores = Q @ K.T
attention_scores

tensor([[0.6652, 0.5032, 0.6717, 1.2272, 1.1343, 1.0683],
        [0.9317, 0.7706, 0.9318, 1.7524, 1.6522, 1.4925],
        [0.6386, 0.4676, 0.6233, 1.1683, 1.0654, 0.9890],
        [1.4481, 1.1286, 1.4558, 2.6884, 2.5006, 2.3208],
        [1.5569, 1.2390, 1.5552, 2.9029, 2.7108, 2.4833],
        [1.0831, 0.8051, 1.0580, 1.9877, 1.8193, 1.6806]])

In [35]:
# Step 4 - Scale the attention_scores
scaled_attention_scores = attention_scores  / (K.size()[-1] ** 0.5)
scaled_attention_scores

tensor([[0.3840, 0.2905, 0.3878, 0.7085, 0.6549, 0.6168],
        [0.5379, 0.4449, 0.5380, 1.0118, 0.9539, 0.8617],
        [0.3687, 0.2700, 0.3599, 0.6745, 0.6151, 0.5710],
        [0.8361, 0.6516, 0.8405, 1.5521, 1.4437, 1.3399],
        [0.8989, 0.7153, 0.8979, 1.6760, 1.5651, 1.4337],
        [0.6253, 0.4648, 0.6108, 1.1476, 1.0504, 0.9703]])

In [ ]:
# Step 5 - Calculate the attention weights
attention_weights = F.softmax(scaled_attention_scores, dim = -1)
attention_weights

tensor([[0.1455, 0.1325, 0.1461, 0.2013, 0.1908, 0.1837],
        [0.1348, 0.1229, 0.1349, 0.2166, 0.2044, 0.1864],
        [0.1480, 0.1341, 0.1467, 0.2009, 0.1893, 0.1811],
        [0.1194, 0.0993, 0.1200, 0.2444, 0.2193, 0.1977],
        [0.1155, 0.0961, 0.1154, 0.2512, 0.2248, 0.1971],
        [0.1340, 0.1141, 0.1320, 0.2258, 0.2049, 0.1892]])

In [ ]:
# Step 6 - Calculate the context vector
context_vectors = attention_weights @ V
context_vectors

tensor([[0.6217, 1.0877, 1.0387],
        [0.6312, 1.1069, 1.0566],
        [0.6197, 1.0846, 1.0354],
        [0.6521, 1.1405, 1.0854],
        [0.6551, 1.1475, 1.0918],
        [0.6374, 1.1146, 1.0616]])

In [40]:
from src.attention import SelfAttention

X = torch.rand(6, 3)
attn = SelfAttention(3, 3)
context = attn(X)
context.shape

torch.Size([6, 3])

### Causal Attention

In [17]:
# Input - 6 words, each a 3-dim embedding
X = torch.rand(6, 3)
X

tensor([[0.5779, 0.9040, 0.5547],
        [0.3423, 0.6343, 0.3644],
        [0.7104, 0.9464, 0.7890],
        [0.2814, 0.7886, 0.5895],
        [0.7539, 0.1952, 0.0050],
        [0.3068, 0.1165, 0.9103]])

In [18]:
# Step 1 - Define the query, key and value metrics
w_q = torch.rand(3, 3)
w_k = torch.rand(3, 3)
w_v = torch.rand(3, 3)

print(f"dimension of w_q: {w_q.size()}")

dimension of w_q: torch.Size([3, 3])


In [19]:
# Step 2 - Produce the Q, K and V
Q = X @ w_q
K = X @ w_k
V = X @ w_v

print(f"dimension of Q is : {Q.size()}")
print(f"dimension of K is : {K.size()}")
print(f"dimension of V is : {V.size()}")

dimension of Q is : torch.Size([6, 3])
dimension of K is : torch.Size([6, 3])
dimension of V is : torch.Size([6, 3])


In [20]:
# Step 3 - Calculate the attention scores
attention_scores = Q @ K.T
attention_scores

tensor([[3.3537, 2.2040, 4.1014, 2.7938, 1.4718, 2.5092],
        [2.1944, 1.4420, 2.6841, 1.8278, 0.9639, 1.6432],
        [3.9903, 2.6235, 4.8751, 3.3252, 1.7450, 2.9684],
        [2.6134, 1.7182, 3.1928, 2.1771, 1.1447, 1.9433],
        [1.8417, 1.2095, 2.2569, 1.5350, 0.8095, 1.3957],
        [1.9756, 1.3033, 2.3960, 1.6498, 0.8429, 1.4073]])

In [26]:
# Step 4 - Scale the attention_scores
scaled_attention_scores = attention_scores  / (K.size()[-1] ** 0.5)
scaled_attention_scores

tensor([[1.9363, 1.2725, 2.3679, 1.6130, 0.8497, 1.4487],
        [1.2669, 0.8325, 1.5497, 1.0553, 0.5565, 0.9487],
        [2.3038, 1.5147, 2.8146, 1.9198, 1.0075, 1.7138],
        [1.5089, 0.9920, 1.8434, 1.2570, 0.6609, 1.1220],
        [1.0633, 0.6983, 1.3031, 0.8863, 0.4674, 0.8058],
        [1.1406, 0.7525, 1.3834, 0.9525, 0.4866, 0.8125]])

In [27]:
# Step 5 - Create the mask
mask = torch.triu(torch.ones(X.shape[0], X.shape[0]), diagonal=1)
mask

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])

In [30]:
# Step 6 - Mask the attention scores
masked_attn_scores = scaled_attention_scores.masked_fill(mask.bool(), float('-inf'))
masked_attn_scores

tensor([[1.9363,   -inf,   -inf,   -inf,   -inf,   -inf],
        [1.2669, 0.8325,   -inf,   -inf,   -inf,   -inf],
        [2.3038, 1.5147, 2.8146,   -inf,   -inf,   -inf],
        [1.5089, 0.9920, 1.8434, 1.2570,   -inf,   -inf],
        [1.0633, 0.6983, 1.3031, 0.8863, 0.4674,   -inf],
        [1.1406, 0.7525, 1.3834, 0.9525, 0.4866, 0.8125]])

In [31]:
# Step 7 - Calculate the attention weights
attention_weights = F.softmax(masked_attn_scores, dim = -1)
attention_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6069, 0.3931, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3204, 0.1456, 0.5340, 0.0000, 0.0000, 0.0000],
        [0.2652, 0.1582, 0.3705, 0.2061, 0.0000, 0.0000],
        [0.2297, 0.1594, 0.2919, 0.1924, 0.1266, 0.0000],
        [0.1991, 0.1351, 0.2538, 0.1650, 0.1035, 0.1434]])

In [32]:
# Step 8 - Calculate the context vector
context_vectors = attention_weights @ V
context_vectors

tensor([[1.0420, 0.7062, 0.7715],
        [0.8888, 0.6123, 0.6786],
        [1.1422, 0.7959, 0.7836],
        [1.0316, 0.7521, 0.7512],
        [0.9767, 0.6572, 0.6653],
        [0.9732, 0.6933, 0.6171]])

In [34]:
from src.attention import SelfAttention

X = torch.rand(6, 3)
attn = SelfAttention(3, 3, causal=False)
context = attn(X)
context.shape

torch.Size([6, 3])

In [35]:
attn = SelfAttention(3, 3, causal=True)
X = torch.rand(6, 3)

# reproduce the internals to see the weights
Q = attn.w_q(X)
K = attn.w_k(X)
scores = Q @ K.transpose(-2, -1) / (K.shape[-1] ** 0.5)

seq_len = scores.shape[-1]
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
scores = scores.masked_fill(mask, float('-inf'))

weights = torch.softmax(scores, dim=-1)
print(weights)
print("row sums:", weights.sum(dim=-1))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5289, 0.4711, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3394, 0.3187, 0.3419, 0.0000, 0.0000, 0.0000],
        [0.2539, 0.2413, 0.2572, 0.2477, 0.0000, 0.0000],
        [0.2090, 0.1842, 0.2082, 0.1860, 0.2126, 0.0000],
        [0.1711, 0.1578, 0.1749, 0.1649, 0.1706, 0.1606]],
       grad_fn=<SoftmaxBackward0>)
row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)
